In [3]:
import pandas as pd
path = 'C:\\Users\\oscar\\big_ahh_files\\nba_transactions_2016_2025(1).csv'
df = pd.read_csv(path)
df.shape

(27193, 5)

In [5]:
df.head()

,Date,Team,Acquired,Relinquished,Notes
0,2016-01-01,76ers,NaN,• Hollis Thompson,corneal abrasion in right eye (DTD)
1,2016-01-01,Hornets,NaN,• Al Jefferson,placed on IL recovering from surgery on right ...
2,2016-01-01,Knicks,NaN,• Cleanthony Early,placed on IL with knee injury (from gunshot wo...
3,2016-01-01,Lakers,NaN,• Kobe Bryant,placed on IL with sore right shoulder
4,2016-01-01,Lakers,• Metta World Peace / Ron Artest,NaN,activated from IL


In [7]:
df.tail()

,Date,Team,Acquired,Relinquished,Notes
27188,2025-10-24,Pistons,NaN,• Caris LeVert,placed on IL with hamstring injury
27189,2025-10-24,Raptors,• Ja'Kobe Walter,NaN,returned to lineup
27190,2025-10-24,Rockets,• Isaiah Crawford,NaN,returned to lineup
27191,2025-10-24,Rockets,• Jae'Sean Tate,NaN,returned to lineup
27192,2025-10-24,Warriors,• Moses Moody,NaN,activated from IL


In [9]:
r = df['Relinquished']
a = df['Acquired']
non_null = []
for name in r:
    if type(name) == str and name not in non_null:
        non_null.append(name)
for name in a:
    if type(name) == str and name not in non_null:
        non_null.append(name)

In [11]:
no_dot = [0] * len(non_null)
i = 0
for name in non_null:
    no_dot[i] = non_null[i][2:]
    i += 1

In [13]:
len(no_dot) #list of all names of athletes that were removed or placed on the injury list in the period

1249

In [15]:
#next match the names to the name in the IDs table
path = 'C:\\Users\\oscar\\OneDrive\\Desktop\\HON 492\\nba_api_hoopR_IDs.csv'
id_table = pd.read_csv(path)
id_table.head()

,Unnamed: 0,hoopR_ID,athlete,nba_api_ID
0,0,136,Vince Carter,1713
1,1,165,Jamal Crawford,2037
2,2,272,Manu Ginobili,1938
3,3,609,Dirk Nowitzki,1717
4,4,841,Jason Terry,1891


In [17]:
rel_athletes = id_table['athlete']

match_names = [name for name in rel_athletes if name in no_dot]
unique_id = [name for name in rel_athletes if name not in no_dot]
unique_pst = [name for name in no_dot if name not in rel_athletes.values]

print('Number of athlete name matches: ', len(match_names))
print('Unique athlete names in ID table: ', len(unique_id))
print('Unique athlete names in pro sports transactions table: ', len(unique_pst))

Number of athlete name matches:  984
Unique athlete names in ID table:  215
Unique athlete names in pro sports transactions table:  265


In [19]:
for i in range(len(unique_pst) - len(unique_id)): #make the length of the two lists equal to create DataFrame
    unique_id.append('z')
    
unique_id.sort()
unique_pst.sort()
uniques = pd.DataFrame({'id':unique_id, 'pst':unique_pst})
path = 'C:\\Users\\oscar\\OneDrive\\Desktop\\HON 492\\Excel Analysis\\pst_id_names(1).xlsx'
#uniques.to_excel(path, index=False)

In [21]:
#cross reference against hoopR database and basketball reference to sort players in excel
path = 'C:\\Users\\oscar\\OneDrive\\Desktop\\HON 492\\Excel Analysis\\pst_id_names(out).xlsx'
mismatch_matched = pd.read_excel(path)
mismatch_matched.head()

,id,pst
0,Mike Scott,(James) Mike Scott
1,Tony Parker,(William) Tony Parker
2,AJ Johnson,A.J. Johnson
3,Sasha Vezenkov,Aleksandar Vezenkov / Sasha Vezenkov
4,Brandon Boston,Brandon Boston Jr. / B.J. Boston Jr.


In [23]:
rows, cols = mismatch_matched.shape
print('Athlete matched from unique names of each dataset: ', rows)

#these athletes were sorted out because they played in earlier seasons, later seasons, or played fewer than 10 games in the relevant period
unique_pst = [name for name in unique_pst if name != 'z'] #delete the 'z' items in the list from earlier
print('Athletes that can be removed from the pro sports transactions dataset: ', len(unique_pst) - rows)

#these athletes just didn't get injured in the time period
unique_id = [name for name in unique_id if name != 'z'] #delete the 'z' items in the list from earlier
print('Athletes in the hoopR dataset but not in the pro sports transactions dataset: ', len(unique_id) - rows) #they just didn't get injured

Athlete matched from unique names of each dataset:  83
Athletes that can be removed from the pro sports transactions dataset:  182
Athletes in the hoopR dataset but not in the pro sports transactions dataset:  132


In [25]:
#change the names in the pro sports transactions dataset to match the names in the hoopR datas

#first remove those dots from the string names for the athletes for matching to mismatch_matched information
a = []
for name in df['Acquired']:
    if type(name) == str:
        a.append(name[2:])
    else:
        a.append(name)

r = []
for name in df['Relinquished']:
    if type(name) == str:
        r.append(name[2:])
    else:
        r.append(name)

df['Acquired'] = a
df['Relinquished'] = r
df.head()

,Date,Team,Acquired,Relinquished,Notes
0,2016-01-01,76ers,NaN,Hollis Thompson,corneal abrasion in right eye (DTD)
1,2016-01-01,Hornets,NaN,Al Jefferson,placed on IL recovering from surgery on right ...
2,2016-01-01,Knicks,NaN,Cleanthony Early,placed on IL with knee injury (from gunshot wo...
3,2016-01-01,Lakers,NaN,Kobe Bryant,placed on IL with sore right shoulder
4,2016-01-01,Lakers,Metta World Peace / Ron Artest,NaN,activated from IL


In [27]:
#investigate the rows containing weird names
df[df['Acquired'] == 'fractured rib (out indefinitely)'].head()

,Date,Team,Acquired,Relinquished,Notes


In [29]:
df[df['Acquired'] == 'illness (DTD)'].head() #from my analysis this is just a duplicate record for Terry Rozier's return one day later.

,Date,Team,Acquired,Relinquished,Notes
26770,2025-04-04,Heat,illness (DTD),NaN,returned to lineup


In [31]:
df.drop(index=26770, inplace=True)

In [33]:
df[df['Acquired'] == 'sore lower back (DTD)'].head() #this is a mistake, the name should be Alperen Senguin.

,Date,Team,Acquired,Relinquished,Notes
26160,2025-03-04,Rockets,sore lower back (DTD),NaN,returned to lineup


In [35]:
df.loc[26160, 'Acquired'] = 'Alperen Sengun'

In [37]:
df[df['Acquired'] == 'torn ACL in left knee (out for season)'].head()

,Date,Team,Acquired,Relinquished,Notes


In [39]:
df[df['Relinquished'] == 'fractured rib (out indefinitely)'].head() #this is clearly anthony davis

,Date,Team,Acquired,Relinquished,Notes
25549,2025-01-25,Mavericks,NaN,fractured rib (out indefinitely),placed on IL with fractured rib


In [41]:
df.loc[25549, 'Relinquished'] = 'Anthony Davis'

In [43]:
df[df['Relinquished'] == 'illness (DTD)'].head()

,Date,Team,Acquired,Relinquished,Notes


In [45]:
df[df['Relinquished'] == 'sore lower back (DTD)'].head()

,Date,Team,Acquired,Relinquished,Notes


In [47]:
df[df['Relinquished'] == 'torn ACL in left knee (out for season)'].head() #just a duplicate of Kyrie Irving's knee injury

,Date,Team,Acquired,Relinquished,Notes
26155,2025-03-04,Mavericks,NaN,torn ACL in left knee (out for season),placed on IL with torn ACL in left knee (out f...


In [49]:
df.drop(index=26155, inplace=True)

In [51]:
#next remove the records of players that are not relevant

#names I want to keep from pro sports transactions: the matching names across both tables and the names in mismatched_match
names_to_keep = match_names
print('names in match_names: ', len(match_names))
for name in list(mismatch_matched['pst']):
    names_to_keep.append(name)

#go through row indices of the pro sports transactions data and if the name in the Acquired or Relinquished column is not in the names I want
#to keep, drop that row
ls = [name for name in no_dot if name not in names_to_keep]
df = df[~df['Acquired'].isin(ls)]
df = df[~df['Relinquished'].isin(ls)]

print('names in names_to_keep: ', len(names_to_keep)) #will be 804 + 69 total athletes
print(df.shape)

names in match_names:  984
names in names_to_keep:  1067
(25980, 5)


In [53]:
#next, change the names in the pro sports transactions data to names that match the IDs table

#define function that returns the corresponding ID table name of a pro sports transactions dataset name
def get_id_from_pst(name):
    target_index = mismatch_matched.index[mismatch_matched['pst'] == name]
    idx = int(mismatch_matched.index[mismatch_matched['pst'] == name].values)
    return mismatch_matched.loc[target_index, 'id'].loc[idx]
    
#iterates through all the mismatched names in pro sports transactions and changes all records containing each name
for old_name in mismatch_matched['pst']:
    new_name = get_id_from_pst(old_name)
    locations = df[df['Acquired'] == old_name].index
    df.loc[locations, 'Acquired'] = new_name
    locations = df[df['Relinquished'] == old_name].index
    df.loc[locations, 'Relinquished'] = new_name

C:\Users\oscar\AppData\Local\Temp\ipykernel_2756\1975615436.py:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  idx = int(mismatch_matched.index[mismatch_matched['pst'] == name].values)
C:\Users\oscar\AppData\Local\Temp\ipykernel_2756\1975615436.py:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  idx = int(mismatch_matched.index[mismatch_matched['pst'] == name].values)
C:\Users\oscar\AppData\Local\Temp\ipykernel_2756\1975615436.py:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated 

In [55]:
ls = [name for name in df['Acquired'].unique()]
for name in df['Relinquished'].unique():
    if name not in ls:
        ls.append(name)
len(ls)

1067

In [57]:
ls

[nan,
 'Bruno Caboclo',
 'Kyrie Irving',
 'Jeremy Lin',
 'Willie Cauley-Stein',
 'Justin Anderson',
 'Danilo Gallinari',
 'Jusuf Nurkic',
 'Ian Mahinmi',
 'Jon Leuer',
 'Stephen Curry',
 'Harrison Barnes',
 'Justise Winslow',
 'Tarik Black',
 'Damian Lillard',
 'Delon Wright',
 'Norman Powell',
 'Kevon Looney',
 'Derrick Rose',
 'Tim Hardaway Jr.',
 'Devin Harris',
 'Luis Montero',
 'RJ Hunter',
 'Hassan Whiteside',
 'Salah Mejri',
 'George Hill',
 'Tyreke Evans',
 'Kevin Durant',
 'Nene ',
 'Justin Holiday',
 'Rudy Gobert',
 'Rajon Rondo',
 "D'Angelo Russell",
 'CJ McCollum',
 'Dirk Nowitzki',
 'Zaza Pachulia',
 'Tony Parker',
 'LaMarcus Aldridge',
 'James Ennis III',
 'Miles Plumlee',
 'Tyler Ennis',
 'Avery Bradley',
 'Nicolas Batum',
 'Rodney Hood',
 'Emmanuel Mudiay',
 'Dwight Howard',
 'Joakim Noah',
 'Tyler Johnson',
 'Courtney Lee',
 'Marcus Morris Sr.',
 'Joe Harris',
 'Thabo Sefolosha',
 'Omri Casspi',
 'John Jenkins',
 'Jameer Nelson',
 'Anthony Davis',
 'Bradley Beal',
 'Cr

In [59]:
i = 0
for name in ls:
    if type(name) is float:
        nan_index = i
    i += 1
ls.pop(nan_index)
len(ls) # now list only contains the 1066 player names I want

1066

In [61]:
#the indices of the list ls will serve as the ID for the players in the pro sports transactions dataset
def get_index(value, ls):
    i = 0
    for item in ls:
        if item == value:
            return i
        i += 1

In [63]:
#add a player_id column to make it easier to isolate a single player's records
df['player_id'] = None

for index, row in df.iterrows():
    if type(row['Acquired']) == str:
        df.loc[index, 'player_id'] = get_index(row['Acquired'], ls) 
    else:
        df.loc[index, 'player_id'] = get_index(row['Relinquished'], ls) 

In [65]:
df.head()

,Date,Team,Acquired,Relinquished,Notes,player_id
1,2016-01-01,Hornets,NaN,Al Jefferson,placed on IL recovering from surgery on right ...,133
5,2016-01-01,Mavericks,NaN,Devin Harris,back spasms (DTD),19
6,2016-01-01,Mavericks,NaN,Justin Anderson,placed on IL,4
12,2016-01-01,Raptors,Bruno Caboclo,NaN,activated from IL,0
15,2016-01-02,Cavaliers,Kyrie Irving,NaN,activated from IL,1


In [67]:
pst_key = pd.DataFrame(columns=['pst_name', 'pst_id'])
for index, row in df.iterrows():
    if type(row['Acquired']) == str:
        if row['Acquired'] in pst_key['pst_name'].values:
            continue
        else:
            new_df = pd.DataFrame({'pst_name':[row['Acquired']], 'pst_id':[row['player_id']]})
            pst_key = pd.concat([pst_key, new_df])
    else:
        if row['Relinquished'] in pst_key['pst_name'].values:
            continue
        else:
            new_df = pd.DataFrame({'pst_name':[row['Relinquished']], 'pst_id':[row['player_id']]})
            pst_key = pd.concat([pst_key, new_df])

pst_key.shape

(1071, 2)

In [69]:
pst_key

,pst_name,pst_id
0,Al Jefferson,133
0,Devin Harris,19
0,Justin Anderson,4
0,Bruno Caboclo,0
0,Kyrie Irving,1
...,...,...
0,Quinten Post,1029
0,Jaylen Wells,1063
0,Alex Reese,1030
0,Reece Beekman,1064


In [71]:
len(pst_key['pst_name'].unique())

1067

In [73]:
#find the names that are null values and assign their value to the variable null_value
ls = []
for name in pst_key['pst_name']:
    if name not in ls:
        ls.append(name)
    else:
        null_value = name

In [75]:
#eliminate the rows that contain the null value
pst_key = pst_key[pst_key['pst_name'] != null_value]
len(pst_key['pst_name'].unique())

1067

In [77]:
len(pst_key['pst_id'].unique())

1067

In [79]:
empty = []
for item in pst_key['pst_id'].values:
    if item in empty:
        print(item)
    else:
        empty.append(item)
        null_value = item

None
None
None
None


In [81]:
pst_key[pst_key['pst_id'] == null_value]

,pst_name,pst_id
0,Quenton Jackson,1065


In [83]:
empty = []
for item in pst_key['pst_name'].values:
    if item in empty:
        print(item)
    else:
        empty.append(item)

nan
nan
nan
nan


In [85]:
pst_key = pst_key.dropna()
pst_key.shape

(1066, 2)

In [87]:
ls = [name for name in id_table['athlete'] if name not in pst_key['pst_name'].values]
ls[0]

'Nick Collison'

In [89]:
#print the names of the athletes that weren't injured in the relevant period
ls

['Nick Collison',
 'Josh Smith',
 'Damien Wilkins',
 'Marreese Speights',
 'Jordan Crawford',
 'Jimmer Fredette',
 'Jeremy Pargo',
 'Xavier Silas',
 'Marquis Teague',
 'Scott Machado',
 'Andre Ingram',
 'Aaron Jackson',
 'Jonathan Gibson',
 'Kalin Lucas',
 'James Nunnally',
 'Mike James',
 'Kyle Collinsworth',
 'Walt Lemon Jr.',
 'Julian Washburn',
 'Dairis Bertans',
 'Jordan Loyd',
 'Jaron Blossomgame',
 'Rodney Purvis',
 'Gabe York',
 'Josh Gray',
 'Joe Chealey',
 'Devon Hall',
 'George King',
 'Johnathan Williams',
 'Troy Caupain',
 'Amida Brimah',
 'Donte Grantham',
 'Xavier Cooks',
 'Elijah Bryant',
 'Jared Terrell',
 'Yante Maten',
 'Angel Delgado',
 'J.P. Macura',
 'Vincent Edwards',
 'Bonzie Colson',
 'Marial Shayok',
 'Kevin Hervey',
 'Mikal Bridges',
 'Tariq Owens',
 'Thomas Welsh',
 'Brandon Sampson',
 'Jeremiah Martin',
 'Deng Adel',
 'Rayjon Tucker',
 'Justin Wright-Foreman',
 'Jack McVeigh',
 'Josh Reaves',
 'Isaac Humphries',
 'Marcus Derrickson',
 'Marques Bolden',
 'Ky

In [91]:
pst_key[pst_key['pst_name'] == 'Stephen Curry']

,pst_name,pst_id
0,Stephen Curry,9


In [93]:
pst_name_id = {}
for index, row in pst_key.iterrows():
    pst_name_id[row['pst_name']] = row['pst_id']
print(pst_name_id)

{'Al Jefferson': 133, 'Devin Harris': 19, 'Justin Anderson': 4, 'Bruno Caboclo': 0, 'Kyrie Irving': 1, 'Avery Bradley': 40, 'RJ Hunter': 21, 'Nicolas Batum': 41, 'Jeremy Lin': 2, 'Willie Cauley-Stein': 3, 'Danilo Gallinari': 5, 'Jusuf Nurkic': 6, 'Ian Mahinmi': 7, 'Tyreke Evans': 25, 'Donatas Motiejunas': 104, 'Jon Leuer': 8, 'James Michael McAdoo': 151, 'Stephen Curry': 9, 'Harrison Barnes': 10, 'Justise Winslow': 11, 'Jarrett Jack': 343, 'Tarik Black': 12, 'Luis Montero': 20, 'Damian Lillard': 13, 'Joe Harris': 49, 'Mike Conley': 67, 'Hassan Whiteside': 22, 'Omri Casspi': 51, "D'Angelo Russell": 31, 'Elfrid Payton': 57, 'George Hill': 24, 'DeMarre Carroll': 209, 'Delon Wright': 14, 'Norman Powell': 15, 'Tony Parker': 35, 'Kevin Durant': 26, 'Kevon Looney': 16, 'Jerryd Bayless': 58, 'Derrick Rose': 17, 'Justin Holiday': 28, 'Tim Hardaway Jr.': 18, 'Rajon Rondo': 30, 'CJ McCollum': 32, 'Josh Richardson': 246, 'Dirk Nowitzki': 33, 'Wesley Matthews': 61, 'Zaza Pachulia': 34, 'Salah Mejri

In [95]:
id_table['pst_ID'] = None
for index, row in id_table.iterrows():
    try:
        id_table.loc[index, 'pst_ID'] = pst_name_id[row['athlete']]
    except:
        id_table.loc[index, 'pst_ID'] = -1

id_table

,Unnamed: 0,hoopR_ID,athlete,nba_api_ID,pst_ID
0,0,136,Vince Carter,1713,184
1,1,165,Jamal Crawford,2037,458
2,2,272,Manu Ginobili,1938,165
3,3,609,Dirk Nowitzki,1717,33
4,4,841,Jason Terry,1891,252
...,...,...,...,...,...
1194,1194,4277916,RJ Nembhard Jr.,1630612,747
1195,1195,5211176,Tidjane Salaun,1642275,982
1196,1196,3102532,Vasilije Micic,203995,946
1197,1197,4578893,Vit Krejci,1630249,822


In [97]:
#this result matches what we see on pst and roughly matches injuries reported on fox sports
df[df['player_id'] == 9].head(45)

,Date,Team,Acquired,Relinquished,Notes,player_id
42,2016-01-02,Warriors,Stephen Curry,NaN,returned to lineup,9
830,2016-03-01,Warriors,NaN,Stephen Curry,left ankle injury (DTD),9
853,2016-03-03,Warriors,Stephen Curry,NaN,returned to lineup,9
1608,2016-04-16,Warriors,NaN,Stephen Curry,sprained right ankle (DTD),9
1638,2016-04-18,Warriors,NaN,Stephen Curry,placed on IL with sprained right ankle,9
1668,2016-04-24,Warriors,Stephen Curry,NaN,activated from IL,9
1674,2016-04-25,Warriors,NaN,Stephen Curry,sprained MCL in right knee (DTD),9
1687,2016-04-27,Warriors,NaN,Stephen Curry,placed on IL with sprained MCL in right knee,9
1706,2016-05-09,Warriors,Stephen Curry,NaN,activated from IL,9
3461,2017-01-29,Warriors,NaN,Stephen Curry,placed on IL with stomach flu,9


In [99]:
df.shape

(25980, 6)

In [101]:
test = df[df['Relinquished'].isnull()]
test = test[test['Acquired'].isnull()]
test.head()

,Date,Team,Acquired,Relinquished,Notes,player_id
10217,2019-03-16,Nets,NaN,NaN,activated from IL,None
16466,2021-12-07,Blazers,NaN,NaN,collapsed right lung (out indefinitely),None
24046,2024-06-28,Pacers,NaN,NaN,NaN,None
24959,2024-12-23,NaN,NaN,NaN,NaN,None
26724,2025-04-02,Knicks,NaN,NaN,activated from IL,None


In [157]:
#analysis concludes that the records from the Nets is just a meaningless duplicate
df.drop(10217, axis=0, inplace=True)

#blazers record: duplicate for CJ McCollum
df.drop(16466, axis=0, inplace=True)

#the pacers record is Jalen Smith be activated from the injury list in a trade to the Chicago bulls
df.loc[24046, 'Acquired'] = 'Jalen Smith'
df.loc[24046, 'Notes'] = 'activated from IL'
df.loc[24046, 'player_id'] = 648

#just delete the last one because there is no information to go off of except the date.
df.drop(24959, axis=0, inplace=True)

#Kevin McCullar Jr but he was activated slightly before so just a duplicate
df.drop(26724, axis=0, inplace=True)

In [159]:
test = df[df['Relinquished'].isnull()]
test = test[test['Acquired'].isnull()]
test.head()

,Date,Team,Acquired,Relinquished,Notes,player_id


In [163]:
test = df[df['Relinquished'].notnull()]
test = test[test['Acquired'].notnull()]
test.head()

,Date,Team,Acquired,Relinquished,Notes,player_id
7599,2018-03-23,Hawks,John Collins,John Collins,returned to lineup,439


In [165]:
#easy fix
df.loc[7599, 'Relinquished'] = df.loc[24046, 'Relinquished']

In [169]:
df.loc[7599, :]

Date                    2018-03-23
Team                         Hawks
Acquired              John Collins
Relinquished                   NaN
Notes           returned to lineup
player_id                      439
Name: 7599, dtype: object

In [171]:
path = "C:\\Users\\oscar\\big_ahh_files\\injuries.csv"
df.to_csv(path, index=False)

In [203]:
id_table.drop('Unnamed: 0', axis=1, inplace=True)

In [207]:
id_table.head()

,hoopR_ID,athlete,nba_api_ID,pst_ID
0,136,Vince Carter,1713,184
1,165,Jamal Crawford,2037,458
2,272,Manu Ginobili,1938,165
3,609,Dirk Nowitzki,1717,33
4,841,Jason Terry,1891,252


In [209]:
path = "C:\\Users\\oscar\\OneDrive\\Desktop\\HON 492\\3IDs.csv"
id_table.to_csv(path, index=False)